# Financial News Agentic Workflow System

This notebook implements the three required workflow patterns for a financial analysis system:

1. **Prompt Chaining:** `Ingest News -> Preprocess -> Classify -> Extract -> Summarize`
2. **Routing:** direct each article to an earnings, macro, market, or company-news specialist.
3. **Evaluator-Optimizer:** generate analysis, evaluate quality, and refine weak outputs.

It is designed to run without paid APIs by using bundled CSV data. If you have the Kaggle financial news CSV, a NewsAPI key, or `yfinance` installed, you can plug in live/larger data sources.

## 1. Setup

The chain is intentionally modular: each stage receives structured output from the previous stage and returns structured output to the next stage.

In [1]:
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from html import unescape
import csv
import hashlib
import json
import os
import re
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen, Request
import xml.etree.ElementTree as ET


@dataclass
class NewsArticle:
    title: str
    description: str
    source: str = "unknown"
    published_at: str = ""
    url: str = ""


@dataclass
class ProcessedArticle:
    id: int
    title: str
    text: str
    source: str
    published_at: str
    url: str


SAMPLE_NEWS = [
    NewsArticle(
        title="Nvidia shares rise as analysts lift AI chip revenue forecasts",
        description="Several Wall Street analysts raised price targets for Nvidia after stronger demand checks for data center GPUs.",
        source="Sample Market Wire",
        published_at="2026-09-24T13:10:00Z",
    ),
    NewsArticle(
        title="Apple faces pressure after supplier report points to slower iPhone orders",
        description="A supplier note suggested softer near-term production schedules, weighing on Apple and related hardware names.",
        source="Sample Finance Daily",
        published_at="2026-09-24T15:40:00Z",
    ),
    NewsArticle(
        title="JPMorgan earnings beat estimates as net interest income remains resilient",
        description="The bank reported better-than-expected quarterly earnings, helped by credit quality and stable deposit trends.",
        source="Sample Earnings Desk",
        published_at="2026-09-24T20:05:00Z",
    ),
    NewsArticle(
        title="Fed officials signal caution on rate cuts as inflation progress slows",
        description="Bond yields moved higher after policymakers emphasized that inflation remains above target.",
        source="Sample Macro Brief",
        published_at="2026-09-24T21:15:00Z",
    ),
]

DEFAULT_SAMPLE_CSV = os.path.join("data", "sample_financial_news.csv")
DEFAULT_MARKET_CSV = os.path.join("data", "sample_market_data.csv")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
MEMORY_DIR = Path("memory")
PROMPT_CHAIN_MEMORY_PATH = MEMORY_DIR / "prompt_chain_memory.jsonl"


def log_memory_event(stage, payload):
    """Persist a compact memory record for each prompt-chain stage."""
    MEMORY_DIR.mkdir(exist_ok=True)
    record = {
        "run_id": RUN_ID,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "payload": payload,
    }
    with PROMPT_CHAIN_MEMORY_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record

print(f"Notebook initialized at {datetime.now(timezone.utc).isoformat()} with {len(SAMPLE_NEWS)} built-in fallback articles.")

Notebook initialized at 2026-09-26T05:18:45.750042+00:00 with 4 built-in fallback articles.


## 2. Prompt Chain Stage 1: Ingest News

This stage gathers raw financial news from one of four sources:

1. Kaggle-style CSV file, if `KAGGLE_FINANCIAL_NEWS_CSV` points to a local CSV.
2. NewsAPI.org, if `NEWSAPI_KEY` is available.
3. The bundled `data/sample_financial_news.csv` file included in this repository.
4. Built-in fallback articles, so the notebook remains reproducible even if the CSV is missing.

The notebook is complete without an API key. NewsAPI is only an optional upgrade for live news.

In [2]:
def ingest_from_csv(path, limit=25):
    articles = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            title = row.get("title") or row.get("headline") or row.get("Title") or ""
            description = row.get("description") or row.get("text") or row.get("content") or row.get("Text") or ""
            if title or description:
                articles.append(NewsArticle(
                    title=title,
                    description=description,
                    source=row.get("source") or row.get("Source") or "kaggle_csv",
                    published_at=row.get("publishedAt") or row.get("date") or row.get("Date") or "",
                    url=row.get("url") or row.get("URL") or "",
                ))
            if len(articles) >= limit:
                break
    return articles


def ingest_from_newsapi(query="stock market OR earnings OR Federal Reserve", limit=10):
    api_key = os.getenv("NEWSAPI_KEY")
    if not api_key:
        return []
    params = urlencode({
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": limit,
        "apiKey": api_key,
    })
    request = Request(f"https://newsapi.org/v2/everything?{params}", headers={"User-Agent": "financial-prompt-chain-notebook"})
    with urlopen(request, timeout=20) as response:
        payload = json.loads(response.read().decode("utf-8"))
    articles = []
    for item in payload.get("articles", []):
        articles.append(NewsArticle(
            title=item.get("title") or "",
            description=item.get("description") or item.get("content") or "",
            source=(item.get("source") or {}).get("name", "newsapi"),
            published_at=item.get("publishedAt") or "",
            url=item.get("url") or "",
        ))
    return articles


def ingest_from_yahoo_finance_rss(tickers=None, limit=10):
    tickers = tickers or ["AAPL", "MSFT", "NVDA", "TSLA", "JPM"]
    ticker_query = ",".join(tickers)
    url = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker_query}&region=US&lang=en-US"
    request = Request(url, headers={"User-Agent": "financial-prompt-chain-notebook"})
    with urlopen(request, timeout=20) as response:
        xml_text = response.read().decode("utf-8", errors="ignore")
    root = ET.fromstring(xml_text)
    articles = []
    for item in root.findall(".//item"):
        title = item.findtext("title") or ""
        description = item.findtext("description") or ""
        published_at = item.findtext("pubDate") or ""
        url = item.findtext("link") or ""
        if title or description:
            articles.append(NewsArticle(
                title=title,
                description=description,
                source="Yahoo Finance RSS",
                published_at=published_at,
                url=url,
            ))
        if len(articles) >= limit:
            break
    return articles


def ingest_news(limit=25, source="auto"):
    if source == "yahoo_rss":
        articles = ingest_from_yahoo_finance_rss(limit=limit)
        log_memory_event("ingest", {"source": "Yahoo Finance RSS", "article_count": len(articles)})
        return articles
    csv_path = os.getenv("KAGGLE_FINANCIAL_NEWS_CSV")
    if csv_path and os.path.exists(csv_path):
        articles = ingest_from_csv(csv_path, limit=limit)
        log_memory_event("ingest", {"source": csv_path, "article_count": len(articles)})
        return articles
    newsapi_articles = ingest_from_newsapi(limit=limit)
    if newsapi_articles:
        log_memory_event("ingest", {"source": "NewsAPI", "article_count": len(newsapi_articles)})
        return newsapi_articles
    if os.path.exists(DEFAULT_SAMPLE_CSV):
        articles = ingest_from_csv(DEFAULT_SAMPLE_CSV, limit=limit)
        log_memory_event("ingest", {"source": DEFAULT_SAMPLE_CSV, "article_count": len(articles)})
        return articles
    articles = SAMPLE_NEWS[:limit]
    log_memory_event("ingest", {"source": "built_in_fallback", "article_count": len(articles)})
    return articles


raw_articles = ingest_news(limit=25)
print(f"Ingested {len(raw_articles)} articles")
for article in raw_articles[:3]:
    print("-", article.title)

Ingested 6 articles
- Nvidia shares rise as analysts lift AI chip revenue forecasts
- Apple faces pressure after supplier report points to slower iPhone orders
- JPMorgan earnings beat estimates as net interest income remains resilient


## 3. Prompt Chain Stage 2: Preprocess

This stage cleans article text, removes noise, deduplicates items, and creates normalized records for downstream analysis.

In [3]:
def clean_text(value):
    value = unescape(value or "")
    value = value.replace("�", "'")
    value = re.sub(r"https?://\S+", " ", value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def preprocess_articles(articles):
    processed = []
    seen = set()
    for index, article in enumerate(articles, start=1):
        title = clean_text(article.title)
        description = clean_text(article.description)
        combined = f"{title}. {description}".strip()
        fingerprint = re.sub(r"[^a-z0-9]+", "", combined.lower())[:180]
        if not combined or fingerprint in seen:
            continue
        seen.add(fingerprint)
        processed.append(ProcessedArticle(
            id=len(processed) + 1,
            title=title,
            text=combined,
            source=article.source,
            published_at=article.published_at,
            url=article.url,
        ))
    return processed


processed_articles = preprocess_articles(raw_articles)
log_memory_event("preprocess", {"input_count": len(raw_articles), "output_count": len(processed_articles)})
print(f"Preprocessed {len(processed_articles)} unique articles")
processed_articles[0]

Preprocessed 6 unique articles


ProcessedArticle(id=1, title='Nvidia shares rise as analysts lift AI chip revenue forecasts', text='Nvidia shares rise as analysts lift AI chip revenue forecasts. Several Wall Street analysts raised price targets for Nvidia after stronger demand checks for data center GPUs.', source='Sample Market Wire', published_at='2026-09-24T13:10:00Z', url='')

## 4. Prompt Chain Stage 3: Classify

This stage routes each article into a financial content class. In a production agentic system, this classifier could be an LLM prompt. Here, a transparent keyword classifier makes the notebook runnable offline.

In [4]:
CLASS_KEYWORDS = {
    "earnings": ["earnings", "revenue", "profit", "guidance", "quarter", "estimates", "beat", "miss"],
    "macro": ["fed", "federal reserve", "inflation", "rates", "yield", "gdp", "jobs", "treasury"],
    "market_movement": ["shares", "stock", "rally", "rise", "fall", "selloff", "price target", "analysts"],
    "company_news": ["supplier", "demand", "orders", "product", "merger", "acquisition", "lawsuit"],
}


def classify_article(article):
    text = article.text.lower()
    scores = {
        label: sum(1 for keyword in keywords if keyword in text)
        for label, keywords in CLASS_KEYWORDS.items()
    }
    label, score = max(scores.items(), key=lambda item: item[1])
    if score == 0:
        label = "general_financial_news"
    confidence = min(0.95, 0.45 + (score * 0.15)) if score else 0.35
    return {"label": label, "confidence": round(confidence, 2), "scores": scores}


classified_articles = []
for article in processed_articles:
    classified_articles.append({
        "article": asdict(article),
        "classification": classify_article(article),
    })
log_memory_event("classify", {"article_count": len(classified_articles), "class_mix": dict(Counter(item["classification"]["label"] for item in classified_articles))})

for item in classified_articles:
    print(f"[{item['classification']['label']}] {item['article']['title']}")

[market_movement] Nvidia shares rise as analysts lift AI chip revenue forecasts
[company_news] Apple faces pressure after supplier report points to slower iPhone orders
[earnings] JPMorgan earnings beat estimates as net interest income remains resilient
[macro] Fed officials signal caution on rate cuts as inflation progress slows
[market_movement] Tesla stock falls after margin concerns offset delivery growth
[market_movement] Microsoft gains as cloud demand supports enterprise software outlook


## 5. Prompt Chain Stage 4: Extract

This stage extracts investment-relevant signals: tickers, companies, sentiment, catalysts, and risk terms.

In [5]:
COMPANY_TO_TICKER = {
    "apple": "AAPL",
    "nvidia": "NVDA",
    "microsoft": "MSFT",
    "amazon": "AMZN",
    "tesla": "TSLA",
    "jpmorgan": "JPM",
    "meta": "META",
    "alphabet": "GOOGL",
}

POSITIVE_TERMS = {"rise", "raised", "stronger", "beat", "resilient", "better", "stable", "growth", "rally"}
NEGATIVE_TERMS = {"pressure", "slower", "softer", "weighing", "miss", "risk", "fall", "selloff", "inflation"}
CATALYST_TERMS = {"earnings", "guidance", "price targets", "demand", "orders", "rates", "inflation", "analysts"}
RISK_TERMS = {"inflation", "slower", "pressure", "softer", "rates", "yield", "lawsuit", "miss"}
TICKER_STOPWORDS = {"AI", "API", "CEO", "CFO", "CNBC", "ETF", "EV", "GDP", "IPO", "SEC", "USA", "US"}


def extract_signals(item):
    article = item["article"]
    text = article["text"]
    lower_text = text.lower()
    tickers = sorted({ticker for company, ticker in COMPANY_TO_TICKER.items() if company in lower_text})
    explicit_tickers = re.findall(r"\b[A-Z]{2,5}\b", text)
    tickers = sorted(set(tickers + [ticker for ticker in explicit_tickers if ticker not in TICKER_STOPWORDS]))

    words = re.findall(r"[a-zA-Z]+", lower_text)
    positive_hits = [word for word in words if word in POSITIVE_TERMS]
    negative_hits = [word for word in words if word in NEGATIVE_TERMS]
    sentiment_score = len(positive_hits) - len(negative_hits)
    sentiment = "positive" if sentiment_score > 0 else "negative" if sentiment_score < 0 else "neutral"

    catalysts = sorted(term for term in CATALYST_TERMS if term in lower_text)
    risks = sorted(term for term in RISK_TERMS if term in lower_text)

    return {
        "article_id": article["id"],
        "title": article["title"],
        "class": item["classification"]["label"],
        "class_confidence": item["classification"]["confidence"],
        "tickers": tickers,
        "sentiment": sentiment,
        "sentiment_score": sentiment_score,
        "positive_terms": positive_hits,
        "negative_terms": negative_hits,
        "catalysts": catalysts,
        "risks": risks,
        "source": article["source"],
    }


extracted_signals = [extract_signals(item) for item in classified_articles]
log_memory_event("extract", {"signal_count": len(extracted_signals), "tickers": sorted({ticker for signal in extracted_signals for ticker in signal["tickers"]})})
print(json.dumps(extracted_signals, indent=2))

[
  {
    "article_id": 1,
    "title": "Nvidia shares rise as analysts lift AI chip revenue forecasts",
    "class": "market_movement",
    "class_confidence": 0.95,
    "tickers": [
      "NVDA"
    ],
    "sentiment": "positive",
    "sentiment_score": 3,
    "positive_terms": [
      "rise",
      "raised",
      "stronger"
    ],
    "negative_terms": [],
    "catalysts": [
      "analysts",
      "demand",
      "price targets"
    ],
    "risks": [],
    "source": "Sample Market Wire"
  },
  {
    "article_id": 2,
    "title": "Apple faces pressure after supplier report points to slower iPhone orders",
    "class": "company_news",
    "class_confidence": 0.9,
    "tickers": [
      "AAPL"
    ],
    "sentiment": "negative",
    "sentiment_score": -4,
    "positive_terms": [],
    "negative_terms": [
      "pressure",
      "slower",
      "softer",
      "weighing"
    ],
    "catalysts": [
      "orders"
    ],
    "risks": [
      "pressure",
      "slower",
      "softer"
   

## 6. Prompt Chain Stage 5: Summarize

This final prompt-chain stage turns extracted signals into a concise investment-research brief and records the completed chain in memory.

In [6]:
def summarize_signals(signals):
    by_class = Counter(signal["class"] for signal in signals)
    by_sentiment = Counter(signal["sentiment"] for signal in signals)
    ticker_counts = Counter(ticker for signal in signals for ticker in signal["tickers"])
    catalysts = Counter(catalyst for signal in signals for catalyst in signal["catalysts"])
    risks = Counter(risk for signal in signals for risk in signal["risks"])
    specialists = Counter(signal.get("specialist", "not_routed") for signal in signals)
    market_sources = Counter(
        data.get("source", "unknown")
        for signal in signals
        for data in signal.get("market_data", {}).values()
    )

    top_items = sorted(signals, key=lambda signal: (abs(signal["sentiment_score"]), signal["class_confidence"]), reverse=True)

    brief = []
    brief.append("Financial News Prompt-Chain Brief")
    brief.append("=" * 36)
    brief.append(f"Articles analyzed: {len(signals)}")
    brief.append(f"Content mix: {dict(by_class)}")
    brief.append(f"Sentiment mix: {dict(by_sentiment)}")
    brief.append(f"Most mentioned tickers: {dict(ticker_counts.most_common(5)) or 'none detected'}")
    brief.append(f"Leading catalysts: {dict(catalysts.most_common(5)) or 'none detected'}")
    brief.append(f"Leading risks: {dict(risks.most_common(5)) or 'none detected'}")
    if any(signal.get("specialist") for signal in signals):
        brief.append(f"Specialist routing: {dict(specialists)}")
    if market_sources:
        brief.append(f"Market data sources: {dict(market_sources)}")
    brief.append("")
    brief.append("Key article-level takeaways:")
    for signal in top_items:
        ticker_text = ", ".join(signal["tickers"]) if signal["tickers"] else "broad market"
        catalyst_text = ", ".join(signal["catalysts"]) if signal["catalysts"] else "no explicit catalyst"
        risk_text = ", ".join(signal["risks"]) if signal["risks"] else "no major risk term"
        specialist_text = f" | specialist={signal['specialist']}" if signal.get("specialist") else ""
        brief.append(
            f"- {signal['title']} | class={signal['class']} | tickers={ticker_text}{specialist_text} | "
            f"sentiment={signal['sentiment']} | catalysts={catalyst_text} | risks={risk_text}"
        )
    return "\n".join(brief)


final_summary = summarize_signals(extracted_signals)
log_memory_event("summarize", {"summary_length": len(final_summary), "article_count": len(extracted_signals)})
print(final_summary)

Financial News Prompt-Chain Brief
Articles analyzed: 6
Content mix: {'market_movement': 3, 'company_news': 1, 'earnings': 1, 'macro': 1}
Sentiment mix: {'positive': 3, 'negative': 2, 'neutral': 1}
Most mentioned tickers: {'NVDA': 1, 'AAPL': 1, 'JPM': 1, 'TSLA': 1, 'MSFT': 1}
Leading catalysts: {'analysts': 2, 'demand': 2, 'price targets': 1, 'orders': 1, 'earnings': 1}
Leading risks: {'pressure': 2, 'slower': 1, 'softer': 1, 'inflation': 1, 'yield': 1}

Key article-level takeaways:
- JPMorgan earnings beat estimates as net interest income remains resilient | class=earnings | tickers=JPM | sentiment=positive | catalysts=earnings | risks=no major risk term
- Apple faces pressure after supplier report points to slower iPhone orders | class=company_news | tickers=AAPL | sentiment=negative | catalysts=orders | risks=pressure, slower, softer
- Nvidia shares rise as analysts lift AI chip revenue forecasts | class=market_movement | tickers=NVDA | sentiment=positive | catalysts=analysts, demand

## 7. Prompt Chain Run Report

This report makes the end-to-end chain visually inspectable. It shows stage completion, record counts, extracted tickers, and the memory log path used to persist run metadata.

In [7]:
def build_prompt_chain_report(raw, processed, classified, extracted, summary):
    class_mix = Counter(item["classification"]["label"] for item in classified)
    sentiment_mix = Counter(signal["sentiment"] for signal in extracted)
    tickers = sorted({ticker for signal in extracted for ticker in signal["tickers"]})
    return f"""
# Prompt Chaining Final Report

| Stage | Status | Evidence |
|---|---:|---|
| Ingest News | Complete | {len(raw)} raw articles loaded |
| Preprocess | Complete | {len(processed)} clean, deduplicated articles |
| Classify | Complete | Class mix: {dict(class_mix)} |
| Extract | Complete | Tickers: {tickers or 'none'}; sentiment mix: {dict(sentiment_mix)} |
| Summarize | Complete | {len(summary)} characters in final brief |
| Memory Logging | Complete | `{PROMPT_CHAIN_MEMORY_PATH}` |

## Final Brief

```text
{summary}
```
""".strip()


prompt_chain_report = build_prompt_chain_report(raw_articles, processed_articles, classified_articles, extracted_signals, final_summary)
try:
    from IPython.display import Markdown, display
    display(Markdown(prompt_chain_report))
except Exception:
    print(prompt_chain_report)

# Prompt Chaining Final Report

| Stage | Status | Evidence |
|---|---:|---|
| Ingest News | Complete | 6 raw articles loaded |
| Preprocess | Complete | 6 clean, deduplicated articles |
| Classify | Complete | Class mix: {'market_movement': 3, 'company_news': 1, 'earnings': 1, 'macro': 1} |
| Extract | Complete | Tickers: ['AAPL', 'JPM', 'MSFT', 'NVDA', 'TSLA']; sentiment mix: {'positive': 3, 'negative': 2, 'neutral': 1} |
| Summarize | Complete | 1617 characters in final brief |
| Memory Logging | Complete | `memory\prompt_chain_memory.jsonl` |

## Final Brief

```text
Financial News Prompt-Chain Brief
====================================
Articles analyzed: 6
Content mix: {'market_movement': 3, 'company_news': 1, 'earnings': 1, 'macro': 1}
Sentiment mix: {'positive': 3, 'negative': 2, 'neutral': 1}
Most mentioned tickers: {'NVDA': 1, 'AAPL': 1, 'JPM': 1, 'TSLA': 1, 'MSFT': 1}
Leading catalysts: {'analysts': 2, 'demand': 2, 'price targets': 1, 'orders': 1, 'earnings': 1}
Leading risks: {'pressure': 2, 'slower': 1, 'softer': 1, 'inflation': 1, 'yield': 1}

Key article-level takeaways:
- JPMorgan earnings beat estimates as net interest income remains resilient | class=earnings | tickers=JPM | sentiment=positive | catalysts=earnings | risks=no major risk term
- Apple faces pressure after supplier report points to slower iPhone orders | class=company_news | tickers=AAPL | sentiment=negative | catalysts=orders | risks=pressure, slower, softer
- Nvidia shares rise as analysts lift AI chip revenue forecasts | class=market_movement | tickers=NVDA | sentiment=positive | catalysts=analysts, demand, price targets | risks=no major risk term
- Fed officials signal caution on rate cuts as inflation progress slows | class=macro | tickers=broad market | sentiment=negative | catalysts=inflation | risks=inflation, yield
- Tesla stock falls after margin concerns offset delivery growth | class=market_movement | tickers=TSLA | sentiment=positive | catalysts=no explicit catalyst | risks=pressure
- Microsoft gains as cloud demand supports enterprise software outlook | class=market_movement | tickers=MSFT | sentiment=neutral | catalysts=analysts, demand | risks=no major risk term
```

## 8. Finalized Test Against Real News Data

This test runs the same prompt chain against live Yahoo Finance RSS headlines. It proves the chain works beyond bundled sample data while still avoiding paid APIs or committed secrets.

In [8]:
def run_prompt_chain_only(limit=10, source="auto"):
    raw = ingest_news(limit=limit, source=source)
    processed = preprocess_articles(raw)
    classified = [{"article": asdict(article), "classification": classify_article(article)} for article in processed]
    extracted = [extract_signals(item) for item in classified]
    summary = summarize_signals(extracted)
    log_memory_event("prompt_chain_test", {
        "source": source,
        "raw_count": len(raw),
        "processed_count": len(processed),
        "extracted_count": len(extracted),
        "summary_length": len(summary),
    })
    return {
        "raw": raw,
        "processed": processed,
        "classified": classified,
        "extracted": extracted,
        "summary": summary,
    }


real_news_result = run_prompt_chain_only(limit=10, source="yahoo_rss")
assert len(real_news_result["raw"]) > 0, "Real news ingestion returned no articles"
assert len(real_news_result["processed"]) > 0, "Preprocessing returned no articles"
assert len(real_news_result["classified"]) == len(real_news_result["processed"]), "Classification count mismatch"
assert len(real_news_result["extracted"]) == len(real_news_result["processed"]), "Extraction count mismatch"
assert len(real_news_result["summary"]) > 100, "Summary is too short to be useful"
print("Real-news prompt-chain test passed")
print(real_news_result["summary"])

Real-news prompt-chain test passed
Financial News Prompt-Chain Brief
Articles analyzed: 10
Content mix: {'market_movement': 5, 'general_financial_news': 4, 'company_news': 1}
Sentiment mix: {'neutral': 6, 'positive': 3, 'negative': 1}
Most mentioned tickers: {'TSLA': 4, 'MSFT': 2, 'AVGO': 1, 'NVDA': 1, 'AAPL': 1}
Leading catalysts: none detected
Leading risks: none detected

Key article-level takeaways:
- Fidelity’s Fundamental Large Cap Growth ETF, Explained in Plain English | class=general_financial_news | tickers=broad market | sentiment=positive | catalysts=no explicit catalyst | risks=no major risk term
- Should You Buy Microsoft Stock Now That It's Back Within 6% of Its Record? | class=market_movement | tickers=MSFT | sentiment=positive | catalysts=no explicit catalyst | risks=no major risk term
- NVIDIA (NVDA) vs. Broadcom (AVGO): Which AI Chip Stock Has the Stronger Moat? | class=market_movement | tickers=AVGO, NVDA | sentiment=positive | catalysts=no explicit catalyst | risks=

## 9. Market Data Enrichment with yfinance

News alone is enough for the prompt chain, but routing and evaluation become stronger when article signals are grounded in stock-price and financial context. This stage tries to use `yfinance` for live market data. If `yfinance` is not installed or live data is unavailable, it falls back to the bundled `data/sample_market_data.csv` file so the notebook remains complete and reproducible.

In [9]:
def load_market_data_from_csv(path=DEFAULT_MARKET_CSV):
    market_data = {}
    if not os.path.exists(path):
        return market_data
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ticker = row.get("ticker", "").upper()
            if not ticker:
                continue
            market_data[ticker] = {
                "ticker": ticker,
                "company": row.get("company", ""),
                "current_price": float(row.get("current_price") or 0),
                "previous_close": float(row.get("previous_close") or 0),
                "one_month_return": float(row.get("one_month_return") or 0),
                "average_volume": int(float(row.get("average_volume") or 0)),
                "market_cap": int(float(row.get("market_cap") or 0)),
                "trailing_pe": float(row.get("trailing_pe") or 0),
                "source": row.get("source", "bundled_sample"),
            }
    return market_data


def fetch_yfinance_market_data(tickers):
    try:
        import yfinance as yf
    except Exception:
        return {}

    live_data = {}
    for ticker in tickers:
        try:
            asset = yf.Ticker(ticker)
            history = asset.history(period="1mo")
            info = getattr(asset, "fast_info", {}) or {}
            if history.empty:
                continue
            current_price = float(history["Close"].iloc[-1])
            previous_close = float(history["Close"].iloc[-2]) if len(history) > 1 else current_price
            first_close = float(history["Close"].iloc[0])
            one_month_return = (current_price - first_close) / first_close if first_close else 0
            live_data[ticker] = {
                "ticker": ticker,
                "company": ticker,
                "current_price": round(current_price, 2),
                "previous_close": round(previous_close, 2),
                "one_month_return": round(one_month_return, 4),
                "average_volume": int(history["Volume"].mean()) if "Volume" in history else 0,
                "market_cap": int(info.get("market_cap") or 0),
                "trailing_pe": 0,
                "source": "yfinance",
            }
        except Exception:
            continue
    return live_data


def get_market_context(tickers):
    tickers = sorted(set(tickers))
    fallback_data = load_market_data_from_csv()
    live_data = fetch_yfinance_market_data(tickers)
    return {ticker: live_data.get(ticker) or fallback_data.get(ticker, {"ticker": ticker, "source": "not_available"}) for ticker in tickers}


def enrich_signals_with_market_data(signals):
    tickers = sorted({ticker for signal in signals for ticker in signal["tickers"]})
    market_context = get_market_context(tickers)
    enriched = []
    for signal in signals:
        item = dict(signal)
        item["market_data"] = {ticker: market_context.get(ticker, {"ticker": ticker, "source": "not_available"}) for ticker in signal["tickers"]}
        enriched.append(item)
    return enriched, market_context


market_enriched_signals, market_context = enrich_signals_with_market_data(extracted_signals)
print(json.dumps(market_context, indent=2))

{
  "AAPL": {
    "ticker": "AAPL",
    "company": "Apple",
    "current_price": 246.8,
    "previous_close": 249.2,
    "one_month_return": -0.018,
    "average_volume": 52100000,
    "market_cap": 3660000000000,
    "trailing_pe": 32.1,
    "source": "bundled_sample"
  },
  "JPM": {
    "ticker": "JPM",
    "company": "JPMorgan Chase",
    "current_price": 301.45,
    "previous_close": 296.9,
    "one_month_return": 0.041,
    "average_volume": 9400000,
    "market_cap": 835000000000,
    "trailing_pe": 13.6,
    "source": "bundled_sample"
  },
  "MSFT": {
    "ticker": "MSFT",
    "company": "Microsoft",
    "current_price": 517.35,
    "previous_close": 512.8,
    "one_month_return": 0.036,
    "average_volume": 21600000,
    "market_cap": 3840000000000,
    "trailing_pe": 38.9,
    "source": "bundled_sample"
  },
  "NVDA": {
    "ticker": "NVDA",
    "company": "Nvidia",
    "current_price": 178.25,
    "previous_close": 174.1,
    "one_month_return": 0.082,
    "average_volume": 

## 10. Routing Workflow Pattern

Routing sends each article to the most relevant specialist based on the article class produced earlier in the chain. This mimics an agentic system where a coordinator chooses the right expert for each task.

In [10]:
SPECIALIST_ROUTES = {
    "earnings": "earnings_analyzer",
    "macro": "macro_analyzer",
    "market_movement": "market_analyzer",
    "company_news": "company_news_analyzer",
    "general_financial_news": "general_news_analyzer",
}


def route_signal(signal):
    routed = dict(signal)
    routed["specialist"] = SPECIALIST_ROUTES.get(signal["class"], "general_news_analyzer")
    return routed


routed_signals = [route_signal(signal) for signal in market_enriched_signals]
for signal in routed_signals:
    print(f"{signal['title']} -> {signal['specialist']}")

Nvidia shares rise as analysts lift AI chip revenue forecasts -> market_analyzer
Apple faces pressure after supplier report points to slower iPhone orders -> company_news_analyzer
JPMorgan earnings beat estimates as net interest income remains resilient -> earnings_analyzer
Fed officials signal caution on rate cuts as inflation progress slows -> macro_analyzer
Tesla stock falls after margin concerns offset delivery growth -> market_analyzer
Microsoft gains as cloud demand supports enterprise software outlook -> market_analyzer


## 11. Evaluator-Optimizer Workflow Pattern

This stage generates an initial specialist analysis, evaluates whether it is complete and grounded, and then refines the analysis when the evaluation identifies missing pieces.

In [11]:
def format_market_snapshot(signal):
    snapshots = []
    for ticker, data in signal.get("market_data", {}).items():
        if data.get("source") == "not_available":
            snapshots.append(f"{ticker}: market data unavailable")
            continue
        monthly_return = data.get("one_month_return", 0) * 100
        snapshots.append(
            f"{ticker}: price ${data.get('current_price', 0):.2f}, "
            f"1M return {monthly_return:.1f}%, P/E {data.get('trailing_pe', 0):.1f}, "
            f"source={data.get('source')}"
        )
    return "; ".join(snapshots) if snapshots else "No ticker-level market context available"


def generate_specialist_analysis(signal):
    ticker_text = ", ".join(signal["tickers"]) if signal["tickers"] else "broad market"
    catalyst_text = ", ".join(signal["catalysts"]) if signal["catalysts"] else "no explicit catalyst"
    risk_text = ", ".join(signal["risks"]) if signal["risks"] else "no major risk term detected"
    market_snapshot = format_market_snapshot(signal)
    analysis = (
        f"{signal['specialist']} reviewed '{signal['title']}'. "
        f"Relevant ticker scope: {ticker_text}. Sentiment is {signal['sentiment']}. "
        f"Catalysts: {catalyst_text}. Risks: {risk_text}. Market context: {market_snapshot}."
    )
    return {"signal": signal, "analysis": analysis}


def evaluate_analysis(analysis_item):
    signal = analysis_item["signal"]
    feedback = []
    score = 1.0
    if not signal["tickers"] and signal["class"] != "macro":
        score -= 0.2
        feedback.append("Add ticker or explain why the article is broad-market only.")
    if not signal["catalysts"]:
        score -= 0.15
        feedback.append("Identify a clearer catalyst or state that no explicit catalyst was found.")
    if not signal.get("market_data") and signal["tickers"]:
        score -= 0.25
        feedback.append("Ground the analysis with stock-price or financial data.")
    if signal["class_confidence"] < 0.7:
        score -= 0.1
        feedback.append("Classification confidence is low; qualify the conclusion.")
    score = max(0, round(score, 2))
    return {"quality_score": score, "feedback": feedback or ["Analysis is complete and grounded."]}


def optimize_analysis(analysis_item, evaluation):
    if evaluation["quality_score"] >= 0.85:
        return analysis_item["analysis"]
    additions = " ".join(evaluation["feedback"])
    return f"{analysis_item['analysis']} Refinement: {additions}"


def run_evaluator_optimizer(signals):
    results = []
    for signal in signals:
        generated = generate_specialist_analysis(signal)
        evaluation = evaluate_analysis(generated)
        refined = optimize_analysis(generated, evaluation)
        results.append({
            "title": signal["title"],
            "specialist": signal["specialist"],
            "quality_score": evaluation["quality_score"],
            "feedback": evaluation["feedback"],
            "refined_analysis": refined,
        })
    return results


analysis_results = run_evaluator_optimizer(routed_signals)

print(json.dumps(analysis_results, indent=2))

[
  {
    "title": "Nvidia shares rise as analysts lift AI chip revenue forecasts",
    "specialist": "market_analyzer",
    "quality_score": 1.0,
    "feedback": [
      "Analysis is complete and grounded."
    ],
    "refined_analysis": "market_analyzer reviewed 'Nvidia shares rise as analysts lift AI chip revenue forecasts'. Relevant ticker scope: NVDA. Sentiment is positive. Catalysts: analysts, demand, price targets. Risks: no major risk term detected. Market context: NVDA: price $178.25, 1M return 8.2%, P/E 49.8, source=bundled_sample."
  },
  {
    "title": "Apple faces pressure after supplier report points to slower iPhone orders",
    "specialist": "company_news_analyzer",
    "quality_score": 1.0,
    "feedback": [
      "Analysis is complete and grounded."
    ],
    "refined_analysis": "company_news_analyzer reviewed 'Apple faces pressure after supplier report points to slower iPhone orders'. Relevant ticker scope: AAPL. Sentiment is negative. Catalysts: orders. Risks: pres

## 12. Full Workflow Summary

This optional summary uses the same summarizer after routing and market-data enrichment, so the report includes specialist assignments and market-data sources.

In [12]:
def summarize_signals(signals):
    by_class = Counter(signal["class"] for signal in signals)
    by_sentiment = Counter(signal["sentiment"] for signal in signals)
    ticker_counts = Counter(ticker for signal in signals for ticker in signal["tickers"])
    catalysts = Counter(catalyst for signal in signals for catalyst in signal["catalysts"])
    risks = Counter(risk for signal in signals for risk in signal["risks"])
    specialists = Counter(signal.get("specialist", "not_routed") for signal in signals)
    market_sources = Counter(
        data.get("source", "unknown")
        for signal in signals
        for data in signal.get("market_data", {}).values()
    )

    top_items = sorted(signals, key=lambda signal: (abs(signal["sentiment_score"]), signal["class_confidence"]), reverse=True)

    brief = []
    brief.append("Financial News Prompt-Chain Brief")
    brief.append("=" * 36)
    brief.append(f"Articles analyzed: {len(signals)}")
    brief.append(f"Content mix: {dict(by_class)}")
    brief.append(f"Sentiment mix: {dict(by_sentiment)}")
    brief.append(f"Most mentioned tickers: {dict(ticker_counts.most_common(5)) or 'none detected'}")
    brief.append(f"Leading catalysts: {dict(catalysts.most_common(5)) or 'none detected'}")
    brief.append(f"Leading risks: {dict(risks.most_common(5)) or 'none detected'}")
    brief.append(f"Specialist routing: {dict(specialists)}")
    brief.append(f"Market data sources: {dict(market_sources) or 'none'}")
    brief.append("")
    brief.append("Key article-level takeaways:")
    for signal in top_items:
        ticker_text = ", ".join(signal["tickers"]) if signal["tickers"] else "broad market"
        catalyst_text = ", ".join(signal["catalysts"]) if signal["catalysts"] else "no explicit catalyst"
        risk_text = ", ".join(signal["risks"]) if signal["risks"] else "no major risk term"
        brief.append(
            f"- {signal['title']} | class={signal['class']} | tickers={ticker_text} | "
            f"specialist={signal.get('specialist', 'not_routed')} | sentiment={signal['sentiment']} | "
            f"catalysts={catalyst_text} | risks={risk_text}"
        )
    return "\n".join(brief)


final_summary = summarize_signals(routed_signals)
print(final_summary)

Financial News Prompt-Chain Brief
Articles analyzed: 6
Content mix: {'market_movement': 3, 'company_news': 1, 'earnings': 1, 'macro': 1}
Sentiment mix: {'positive': 3, 'negative': 2, 'neutral': 1}
Most mentioned tickers: {'NVDA': 1, 'AAPL': 1, 'JPM': 1, 'TSLA': 1, 'MSFT': 1}
Leading catalysts: {'analysts': 2, 'demand': 2, 'price targets': 1, 'orders': 1, 'earnings': 1}
Leading risks: {'pressure': 2, 'slower': 1, 'softer': 1, 'inflation': 1, 'yield': 1}
Specialist routing: {'market_analyzer': 3, 'company_news_analyzer': 1, 'earnings_analyzer': 1, 'macro_analyzer': 1}
Market data sources: {'bundled_sample': 5}

Key article-level takeaways:
- JPMorgan earnings beat estimates as net interest income remains resilient | class=earnings | tickers=JPM | specialist=earnings_analyzer | sentiment=positive | catalysts=earnings | risks=no major risk term
- Apple faces pressure after supplier report points to slower iPhone orders | class=company_news | tickers=AAPL | specialist=company_news_analyzer 

## 13. End-to-End Agentic Workflow Function

This wrapper runs the full workflow end to end: prompt chaining, yfinance/market-data enrichment, routing, and evaluator-optimizer refinement.

In [13]:
def run_agentic_workflow(limit=25):
    raw = ingest_news(limit=limit)
    processed = preprocess_articles(raw)
    classified = [{"article": asdict(article), "classification": classify_article(article)} for article in processed]
    extracted = [extract_signals(item) for item in classified]
    enriched, market_context = enrich_signals_with_market_data(extracted)
    routed = [route_signal(signal) for signal in enriched]
    evaluations = run_evaluator_optimizer(routed)
    summary = summarize_signals(routed)
    return {
        "raw_count": len(raw),
        "processed_count": len(processed),
        "classified": classified,
        "extracted_signals": extracted,
        "market_context": market_context,
        "routed_signals": routed,
        "analysis_results": evaluations,
        "summary": summary,
    }


chain_result = run_agentic_workflow(limit=25)
print(chain_result["summary"])
print("\nEvaluator-Optimizer refined analyses:")
for result in chain_result["analysis_results"]:
    print(f"- score={result['quality_score']} | {result['refined_analysis']}")

Financial News Prompt-Chain Brief
Articles analyzed: 6
Content mix: {'market_movement': 3, 'company_news': 1, 'earnings': 1, 'macro': 1}
Sentiment mix: {'positive': 3, 'negative': 2, 'neutral': 1}
Most mentioned tickers: {'NVDA': 1, 'AAPL': 1, 'JPM': 1, 'TSLA': 1, 'MSFT': 1}
Leading catalysts: {'analysts': 2, 'demand': 2, 'price targets': 1, 'orders': 1, 'earnings': 1}
Leading risks: {'pressure': 2, 'slower': 1, 'softer': 1, 'inflation': 1, 'yield': 1}
Specialist routing: {'market_analyzer': 3, 'company_news_analyzer': 1, 'earnings_analyzer': 1, 'macro_analyzer': 1}
Market data sources: {'bundled_sample': 5}

Key article-level takeaways:
- JPMorgan earnings beat estimates as net interest income remains resilient | class=earnings | tickers=JPM | specialist=earnings_analyzer | sentiment=positive | catalysts=earnings | risks=no major risk term
- Apple faces pressure after supplier report points to slower iPhone orders | class=company_news | tickers=AAPL | specialist=company_news_analyzer 

## 14. Why These Workflow Patterns Matter

This project uses prompt chaining to break financial analysis into smaller, reliable steps instead of asking one model or function to do everything at once. Each stage produces structured output that becomes the input for the next stage.

| Chain Step | Purpose | Output |
|---|---|---|
| Ingest News | Pull financial news from NewsAPI, Kaggle CSV, or sample data | Raw articles |
| Preprocess | Clean, normalize, and deduplicate article text | Clean article records |
| Classify | Identify the type of financial content | Earnings, macro, market movement, company news |
| Extract | Pull investment signals from each article | Tickers, sentiment, catalysts, risks |
| Summarize | Convert extracted signals into a research brief | Final financial news summary |

Separating the workflow this way makes the system easier to debug, evaluate, and improve. If one stage produces weak results, that stage can be refined without rewriting the entire pipeline.

## 15. Example Output

The notebook produces a financial news brief that includes:

- Number of articles analyzed
- Content mix by category
- Sentiment breakdown
- Most mentioned tickers
- Leading catalysts
- Leading risks
- Article-level investment takeaways
- Routed specialist for each article
- Market context from yfinance or bundled fallback data
- Evaluator-optimizer quality score and refined analysis

These outputs demonstrate that the chain does more than summarize text: it transforms raw financial news into structured investment signals and then converts those signals into a readable research brief.